IMPORT


In [ ]:
import pandas as pd
import os
from google.colab import drive, files

print(os.getcwd())
drive.mount("/content/drive")

/content
Mounted at /content/drive


This the setup block that loads pandas and os and mounts the Google Drive so that the notebool can access the dataset file stored there. The output above confirms that the Drive has been mounted successfully.

CLEANING PLAN


In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/SE-dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
)

# Drop duplicates
df = df.drop_duplicates()

# Drop duplicate symptom column
df = df.drop(columns=["regurgitation.1"])

# Check class balance after dedup
print(df["diseases"].value_counts())

diseases
cystitis                          1219
nose disorder                     1218
vulvodynia                        1218
complex regional pain syndrome    1217
spondylosis                       1216
                                  ... 
open wound of the cheek              1
open wound of the knee               1
rheumatic fever                      1
rocky mountain spotted fever         1
turner syndrome                      1
Name: count, Length: 773, dtype: int64


After cleaning the data, we found out that the dataset has 773 diseases with a  severe class imbalance. Several diseases like " Cystitis" and "Nose order" had over 1200 samples, while others like "open wound of the cheek" had just 1 sample. This imbalance in the dataset needs to be addressed before modelling.

DROPPING RARE DISEASES AT THE BOTTOM


In [ ]:
# need more samples per split
counts = df["diseases"].value_counts()
df = df[df["diseases"].isin(counts[counts >= 5].index)]

Since many diseases have too few samples to be split into train, validation, and test sets, this section removes any disease class with fewer than 5 samples. It counts the number of rows per disease, then filters the dataset to keep only diseases meeting that minimum threshold. This ensures every remaining class has enough data to be usable during modeling, though the earlier class imbalance still needs to be addressed next. i.e., some disease having far more samples than others

HANDLING CLASS BALANCE (some have ~1200, others 1 or 0)


In [ ]:
# capping samples at 300
df = (
    df.groupby("diseases")
    .apply(lambda x: x.sample(min(len(x), 300)))
    .reset_index(drop=True)
)

/tmp/ipykernel_1137/3400104245.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 300)))


This section is to further handle the class imbalances. To reduce the extreme imbalance between diseases, the dataset is capped at a maximum of 300 samples per disease. This is done by grouping rows by disease and randomly sampling up to 300 from each group. Diseases with fewer than 300 samples keep all their rows, while the larger classes are downsampled. This prevents the model from being dominated by the most frequent diseases while still preserving as much data as possible from the rarer ones.

DROPPING ZERO-SYMPTOM ROWS


In [ ]:
symptom_cols = df.columns[1:]
zero_rows = df[symptom_cols].sum(axis=1) == 0
print("Zero symptom rows: ", zero_rows.sum())
df = df[~zero_rows]

Zero symptom rows:  0


This section checks each row for cases where all symptom columns sum to zero, meaning no symptoms were recorded for that entry. Such rows are useless for training, since there is nothing to learn from them. The check found 0 zero-symptom rows , so no rows needed to be removed at this stage.

SAVING DATA


In [ ]:
print(f"Final shape: {df.shape}")
print(f"Unique diseases: {df['diseases'].nunique()}")
print(df["diseases"].value_counts())

df.to_csv('cleaned_diseases_symptoms.csv', index=False)
files.download('cleaned_diseases_symptoms.csv')

Final shape: (101485, 377)
Unique diseases: 658
diseases
acne                        300
hemangioma                  300
hemiplegia                  300
hemorrhoids                 300
herniated disk              300
                           ... 
pinguecula                    5
pyloric stenosis              5
lymphogranuloma venereum      5
thyroid cancer                5
abscess of the lung           5
Name: count, Length: 658, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In this section, the cleaned dataset is finalized and saved. The final shape is 101,485 rows by 377 columns, covering 658 unique diseases, with sample counts now ranging between 5, which is the minimum threshold and 300, the maximum threshold per disease. The cleaned data is exported to cleaned_diseases_symptoms.csv and downloaded locally for use in the next stage of the pipeline.